# PolicyDelta

Semantic Privacy Policy Change Detector

Run the cells in order using **Runtime → Run all** in Google Colab.

In [ ]:
%pip install -q beautifulsoup4==4.14.3 PyMuPDF==1.26.7 scikit-learn==1.7.2 sentence-transformers==6.2.0 huggingface-hub==1.33.0

## Configuration

In [ ]:
import hashlib
import importlib.util
import json
import platform
import tempfile
import random
import re
import sys
import unittest
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Literal

from bs4 import BeautifulSoup, NavigableString, Tag
import pymupdf


class PolicyDeltaError(ValueError):
    """Invalid policy data or application configuration."""


class EvidenceError(PolicyDeltaError):
    """Source evidence could not be verified."""


@dataclass(frozen=True, slots=True)
class Settings:
    seed: int = 42
    max_document_chars: int = 250_000
    max_pdf_pages: int = 200
    max_pdf_bytes: int = 25_000_000

    def __post_init__(self) -> None:
        if type(self.seed) is not int or not 0 <= self.seed < 2**32:
            raise PolicyDeltaError("Seed must be a nonnegative 32-bit integer.")
        if type(self.max_document_chars) is not int or self.max_document_chars < 1:
            raise PolicyDeltaError("Document character limit must be positive.")
        if type(self.max_pdf_pages) is not int or self.max_pdf_pages < 1:
            raise PolicyDeltaError("PDF page limit must be positive.")
        if type(self.max_pdf_bytes) is not int or self.max_pdf_bytes < 1:
            raise PolicyDeltaError("PDF file size limit must be positive.")


SETTINGS = Settings()


def initialize(settings: Settings = SETTINGS) -> dict[str, str | int]:
    if sys.version_info < (3, 10):
        raise RuntimeError("Python 3.10 or later is required.")
    random.seed(settings.seed)
    try:
        colab = importlib.util.find_spec("google.colab") is not None
    except (ImportError, ModuleNotFoundError, ValueError):
        colab = False
    return {
        "python": platform.python_version(),
        "environment": "colab" if colab else "python",
        "seed": settings.seed,
    }


RUNTIME = initialize()
print(f"PolicyDelta | Python {RUNTIME['python']} | {RUNTIME['environment']}")

## Document model

In [ ]:
@dataclass(frozen=True, slots=True)
class PolicyDocument:
    document_id: str
    role: Literal["old", "new"]
    text: str
    sha256: str

    def __post_init__(self) -> None:
        if not isinstance(self.document_id, str) or not self.document_id.strip():
            raise PolicyDeltaError("Document ID must be nonempty.")
        if self.role not in ("old", "new"):
            raise PolicyDeltaError("Document role must be 'old' or 'new'.")
        if not isinstance(self.text, str) or not self.text.strip():
            raise PolicyDeltaError("Policy text must be nonempty.")
        if self.sha256 != hashlib.sha256(self.text.encode("utf-8")).hexdigest():
            raise EvidenceError("Document checksum does not match its content.")

    @classmethod
    def from_text(
        cls,
        text: str,
        *,
        role: Literal["old", "new"],
        document_id: str | None = None,
        settings: Settings = SETTINGS,
    ) -> "PolicyDocument":
        if not isinstance(text, str):
            raise PolicyDeltaError("Policy content must be text.")
        if len(text) > settings.max_document_chars:
            raise PolicyDeltaError("Document exceeds the configured size limit.")
        digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
        return cls(document_id if document_id is not None else f"{role}-{digest[:16]}",
                   role, text, digest)

    @classmethod
    def from_txt(
        cls,
        path: str | Path,
        *,
        role: Literal["old", "new"],
        document_id: str | None = None,
        settings: Settings = SETTINGS,
    ) -> "PolicyDocument":
        """Read UTF-8 policy text without changing its line endings."""
        try:
            source = Path(path)
        except (TypeError, ValueError) as exc:
            raise PolicyDeltaError("A valid text-file path is required.") from exc
        if source.suffix.lower() != ".txt":
            raise PolicyDeltaError("Only .txt files are supported.")
        try:
            with source.open(encoding="utf-8-sig", newline="") as file:
                text = file.read(settings.max_document_chars + 1)
        except (OSError, UnicodeError) as exc:
            raise PolicyDeltaError(f"Unable to read UTF-8 text from '{source.name}'.") from exc
        return cls.from_text(text, role=role, document_id=document_id, settings=settings)

    @classmethod
    def from_html(
        cls,
        html: str,
        *,
        role: Literal["old", "new"],
        document_id: str | None = None,
        settings: Settings = SETTINGS,
    ) -> "PolicyDocument":
        """Extract readable text from HTML with stable paragraph boundaries."""
        if not isinstance(html, str):
            raise PolicyDeltaError("HTML content must be text.")
        if len(html) > settings.max_document_chars * 8:
            raise PolicyDeltaError("HTML input exceeds the configured size limit.")

        soup = BeautifulSoup(html, "html.parser")
        root = soup.find("main") or soup.find("article") or soup.body or soup
        excluded = {"script", "style", "noscript", "template", "svg", "nav",
                    "header", "footer", "aside", "form"}
        chrome = {"nav", "navigation", "navbar", "menu", "sidebar", "breadcrumb",
                  "cookie-banner", "cookie-notice", "site-header", "site-footer"}
        for tag in list(root.find_all(True)):
            if tag.name is None:
                continue
            identifiers = [str(tag.get("id", "")), *tag.get("class", [])]
            if (tag.name in excluded or tag.has_attr("hidden")
                    or tag.get("aria-hidden") == "true"
                    or re.search(r"(?:display\s*:\s*none|visibility\s*:\s*hidden)",
                                 str(tag.get("style", "")), flags=re.I)
                    or tag.get("role") in {"navigation", "banner", "contentinfo", "search"}
                    or any(value.lower() in chrome for value in identifiers)):
                tag.decompose()

        paragraphs: list[str] = []
        blocks = {"p", "li", "blockquote", "pre", "dt", "dd", "h1", "h2", "h3",
                  "h4", "h5", "h6", "td", "th"}
        containers = blocks | {"div", "section", "article", "main", "ul", "ol",
                               "table", "tbody", "thead", "tr", "dl"}

        def append(text: str) -> None:
            normalized = re.sub(r"\s+", " ", text).strip()
            if normalized:
                paragraphs.append(normalized)

        def collect(tag: Tag) -> None:
            if tag.name in blocks and tag.find(["br", *containers]) is None:
                append(tag.get_text())
                return
            inline: list[str] = []
            for child in tag.children:
                if isinstance(child, NavigableString):
                    inline.append(str(child))
                elif isinstance(child, Tag) and child.name == "br":
                    append("".join(inline))
                    inline.clear()
                elif isinstance(child, Tag) and child.name in containers:
                    append("".join(inline))
                    inline.clear()
                    collect(child)
                elif isinstance(child, Tag):
                    inline.append(child.get_text())
            append("".join(inline))

        collect(root)
        return cls.from_text("\n\n".join(paragraphs), role=role,
                             document_id=document_id, settings=settings)

    @classmethod
    def from_html_file(
        cls,
        path: str | Path,
        *,
        role: Literal["old", "new"],
        document_id: str | None = None,
        settings: Settings = SETTINGS,
    ) -> "PolicyDocument":
        """Read a UTF-8 HTML file and extract its policy text."""
        try:
            source = Path(path)
        except (TypeError, ValueError) as exc:
            raise PolicyDeltaError("A valid HTML-file path is required.") from exc
        if source.suffix.lower() not in {".html", ".htm"}:
            raise PolicyDeltaError("Only .html and .htm files are supported.")
        try:
            with source.open(encoding="utf-8-sig", newline="") as file:
                html = file.read(settings.max_document_chars * 8 + 1)
        except (OSError, UnicodeError) as exc:
            raise PolicyDeltaError(f"Unable to read UTF-8 HTML from '{source.name}'.") from exc
        return cls.from_html(html, role=role, document_id=document_id,
                             settings=settings)

    @classmethod
    def from_pdf(
        cls,
        path: str | Path,
        *,
        role: Literal["old", "new"],
        document_id: str | None = None,
        settings: Settings = SETTINGS,
    ) -> "PolicyDocument":
        """Extract text from an unencrypted, text-based PDF."""
        try:
            source = Path(path)
        except (TypeError, ValueError) as exc:
            raise PolicyDeltaError("A valid PDF path is required.") from exc
        if source.suffix.lower() != ".pdf":
            raise PolicyDeltaError("Only .pdf files are supported.")
        try:
            if source.stat().st_size > settings.max_pdf_bytes:
                raise PolicyDeltaError("PDF exceeds the configured file size limit.")
            with pymupdf.open(filename=str(source), filetype="pdf") as pdf:
                if pdf.needs_pass:
                    raise PolicyDeltaError("Password-protected PDFs are not supported.")
                if pdf.page_count > settings.max_pdf_pages:
                    raise PolicyDeltaError("PDF exceeds the configured page limit.")
                pages: list[str] = []
                length = 0
                for page in pdf:
                    text = page.get_text("text", sort=True).strip()
                    if not text:
                        continue
                    length += len(text) + (2 if pages else 0)
                    if length > settings.max_document_chars:
                        raise PolicyDeltaError("Document exceeds the configured size limit.")
                    pages.append(text)
        except (OSError, pymupdf.FileDataError) as exc:
            raise PolicyDeltaError(f"Unable to read PDF '{source.name}'.") from exc
        if not pages:
            raise PolicyDeltaError("PDF contains no extractable text; image-only PDFs require OCR.")
        return cls.from_text("\n\n".join(pages), role=role,
                             document_id=document_id, settings=settings)

    def excerpt(self, start: int, end: int) -> "EvidenceSpan":
        if (type(start) is not int or type(end) is not int
                or start < 0 or end > len(self.text) or start >= end):
            raise EvidenceError("Evidence offsets are outside the source text.")
        return EvidenceSpan(self.document_id, start, end, self.text[start:end])


@dataclass(frozen=True, slots=True)
class EvidenceSpan:
    document_id: str
    start: int
    end: int
    text: str


def verify_evidence(span: EvidenceSpan, document: PolicyDocument) -> None:
    """Confirm that a cited excerpt is an exact slice of the source."""
    if (span.document_id != document.document_id
            or type(span.start) is not int
            or type(span.end) is not int
            or span.start < 0
            or span.start >= span.end
            or span.end > len(document.text)
            or document.text[span.start:span.end] != span.text):
        raise EvidenceError("Evidence does not match the source document.")
    if hashlib.sha256(document.text.encode("utf-8")).hexdigest() != document.sha256:
        raise EvidenceError("Source document integrity check failed.")


def document_json(document: PolicyDocument) -> str:
    return json.dumps(asdict(document), ensure_ascii=False, indent=2)

## Text preprocessing

In [ ]:
import unicodedata


def normalize_policy_text(text: str) -> str:
    """Produce searchable text without changing the stored source."""
    return unicodedata.normalize("NFC", re.sub(r"\s+", " ", text)).strip()


@dataclass(frozen=True, slots=True)
class PolicyClause:
    clause_id: str
    document_id: str
    text: str
    start: int
    end: int
    section: str | None
    paragraph_index: int

    @property
    def normalized_text(self) -> str:
        return normalize_policy_text(self.text)


def _is_heading(line: str, *, isolated: bool) -> bool:
    label = re.sub(r"^(?:#{1,6}\s+|\d+(?:\.\d+)*[.)]?\s+)", "", line).strip()
    if not label or len(label) > 90 or len(label.split()) > 12:
        return False
    if label.endswith((".", "!", "?")):
        return False
    if line.startswith("#") or label.endswith(":") or label.isupper() or label.istitle():
        return True
    return (isolated and label[0].isupper()
            and not re.match(r"^(?:We|You|They|This|These|It)\b", label)
            and not re.search(r"\b(?:is|are|was|were|may|must|will|can|shall|should)\b",
                              label, flags=re.I))


def _sentence_ranges(text: str):
    abbreviations = {"e.g.", "i.e.", "mr.", "mrs.", "ms.", "dr.",
                     "prof.", "inc.", "ltd.", "etc.", "u.s.", "u.k.", "vs."}
    start = 0
    for match in re.finditer(r"[.!?]+(?:[\"'”’)]*)(?=\s|$)", text):
        end = match.end()
        preceding = text[start:end].rstrip('\"\'”’)')
        last = preceding.split()[-1].lower() if preceding.split() else ""
        following = text[end:].lstrip()
        if (last in abbreviations
                and not (last in {"u.s.", "u.k."}
                         and re.match(r"(?:We|Our|You|They|This)\b", following))):
            continue
        if (re.fullmatch(r"(?:\d{1,2}|[A-Za-z])\.", last)
                or (following and not re.match(r"[A-Z0-9\"'“‘(]", following))):
            continue
        if text[start:end].strip():
            yield start, end
        start = end
    if text[start:].strip():
        yield start, len(text)


def segment_policy(document: PolicyDocument) -> list[PolicyClause]:
    """Split policy text into traceable clauses with section context."""
    clauses: list[PolicyClause] = []
    section: str | None = None
    paragraph: tuple[int, int] | None = None
    paragraph_index = 0

    def flush() -> None:
        nonlocal paragraph, paragraph_index
        if paragraph is None:
            return
        left, right = paragraph
        block = document.text[left:right]
        for local_start, local_end in _sentence_ranges(block):
            raw = block[local_start:local_end]
            leading = len(raw) - len(raw.lstrip())
            trailing = len(raw.rstrip())
            start, end = left + local_start + leading, left + local_start + trailing
            if start < end:
                clauses.append(PolicyClause(
                    f"{document.document_id}:clause:{len(clauses) + 1:05d}",
                    document.document_id, document.text[start:end], start, end,
                    section, paragraph_index,
                ))
        paragraph_index += 1
        paragraph = None

    lines = document.text.splitlines(keepends=True)
    offset = 0
    for index, line in enumerate(lines):
        content_end = offset + len(line.rstrip("\r\n"))
        content = document.text[offset:content_end].strip()
        isolated = (index == 0 or not lines[index - 1].strip()
                    or index == len(lines) - 1 or not lines[index + 1].strip())
        if not content:
            flush()
        elif _is_heading(content, isolated=isolated):
            flush()
            section = re.sub(r"^(?:#{1,6}\s+|\d+(?:\.\d+)*[.)]?\s+)",
                             "", content).rstrip(":").strip()
        elif re.match(r"^(?:[-*•]\s+|\d+[.)]\s+)", content):
            flush()
            paragraph = (offset, content_end)
            flush()
        elif paragraph is None:
            paragraph = (offset, content_end)
        else:
            paragraph = (paragraph[0], content_end)
        offset += len(line)
    flush()
    if not clauses:
        raise PolicyDeltaError("Document contains no analyzable clauses.")
    return clauses


## Clause alignment

In [ ]:
from collections import defaultdict, deque
from functools import lru_cache
import math

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


@dataclass(frozen=True, slots=True)
class ClauseMatch:
    old: PolicyClause
    new: PolicyClause
    similarity: float
    method: Literal["exact", "tfidf", "minilm"]


@dataclass(frozen=True, slots=True)
class ClauseAlignment:
    matches: tuple[ClauseMatch, ...]
    unmatched_old: tuple[PolicyClause, ...]
    unmatched_new: tuple[PolicyClause, ...]


def tfidf_similarity(old: list[PolicyClause], new: list[PolicyClause]) -> np.ndarray:
    """Calculate lexical similarity using a shared word and bigram vocabulary."""
    if not old or not new:
        return np.zeros((len(old), len(new)), dtype=float)
    texts = [clause.normalized_text for clause in [*old, *new]]
    try:
        vectors = TfidfVectorizer(ngram_range=(1, 2), token_pattern=r"(?u)\b\w+\b").fit_transform(texts)
    except ValueError as exc:
        if "empty vocabulary" not in str(exc).lower():
            raise
        return np.zeros((len(old), len(new)), dtype=float)
    return cosine_similarity(vectors[:len(old)], vectors[len(old):])


@lru_cache(maxsize=1)
def load_semantic_model():
    """Load the sentence encoder once per notebook session."""
    try:
        from sentence_transformers import SentenceTransformer
        return SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    except Exception as exc:
        raise PolicyDeltaError("Unable to load MiniLM. Check the installation and model download.") from exc


def minilm_similarity(
    old: list[PolicyClause], new: list[PolicyClause], *, encoder=None,
) -> np.ndarray:
    """Compute cosine similarities between contextual clause embeddings."""
    if not old or not new:
        return np.zeros((len(old), len(new)), dtype=float)
    model = encoder if encoder is not None else load_semantic_model()
    texts = [clause.normalized_text for clause in [*old, *new]]
    try:
        vectors = np.asarray(model.encode(
            texts, batch_size=32, convert_to_numpy=True,
            normalize_embeddings=True, show_progress_bar=False,
        ), dtype=float)
    except Exception as exc:
        raise PolicyDeltaError("Unable to encode policy clauses.") from exc
    if (vectors.ndim != 2 or vectors.shape[0] != len(texts)
            or vectors.shape[1] == 0 or not np.isfinite(vectors).all()):
        raise PolicyDeltaError("The embedding model returned invalid vectors.")
    norms = np.linalg.norm(vectors, axis=1)
    if np.any(norms == 0):
        raise PolicyDeltaError("The embedding model returned a zero vector.")
    unit_vectors = vectors / norms[:, None]
    return np.clip(unit_vectors[:len(old)] @ unit_vectors[len(old):].T, -1.0, 1.0)


def align_policies(
    old: PolicyDocument,
    new: PolicyDocument,
    *,
    minimum_similarity: float | None = None,
    candidates_per_clause: int = 5,
    method: Literal["tfidf", "minilm"] = "tfidf",
    encoder=None,
) -> ClauseAlignment:
    """Align exact clauses and greedily assign distinct similarity matches."""
    if old.role != "old" or new.role != "new":
        raise PolicyDeltaError("Expected old and new documents in that order.")
    if method not in ("tfidf", "minilm"):
        raise PolicyDeltaError("Alignment method must be tfidf or minilm.")
    if minimum_similarity is None:
        minimum_similarity = 0.40 if method == "tfidf" else 0.65
    if (isinstance(minimum_similarity, bool)
            or not isinstance(minimum_similarity, (int, float))
            or not math.isfinite(minimum_similarity)
            or not 0 < minimum_similarity <= 1):
        raise PolicyDeltaError("Minimum similarity must be between 0 and 1.")
    if type(candidates_per_clause) is not int or candidates_per_clause < 1:
        raise PolicyDeltaError("Candidate count must be a positive integer.")

    old_clauses, new_clauses = segment_policy(old), segment_policy(new)
    available: dict[str, deque[int]] = defaultdict(deque)
    for index, clause in enumerate(new_clauses):
        available[clause.normalized_text.casefold()].append(index)

    matches: list[ClauseMatch] = []
    matched_old: set[int] = set()
    matched_new: set[int] = set()
    for index, clause in enumerate(old_clauses):
        positions = available[clause.normalized_text.casefold()]
        if positions:
            new_index = positions.popleft()
            matches.append(ClauseMatch(clause, new_clauses[new_index], 1.0, "exact"))
            matched_old.add(index)
            matched_new.add(new_index)

    old_indices = [i for i in range(len(old_clauses)) if i not in matched_old]
    new_indices = [i for i in range(len(new_clauses)) if i not in matched_new]
    similarity_fn = tfidf_similarity if method == "tfidf" else minilm_similarity
    remaining_old = [old_clauses[i] for i in old_indices]
    remaining_new = [new_clauses[i] for i in new_indices]
    scores = (similarity_fn(remaining_old, remaining_new) if method == "tfidf"
              else similarity_fn(remaining_old, remaining_new, encoder=encoder))
    candidates: list[tuple[float, int, int]] = []
    for row_index, row in enumerate(scores):
        for col_index in np.argsort(-row, kind="stable")[:candidates_per_clause]:
            similarity = float(row[col_index])
            if similarity >= minimum_similarity:
                candidates.append((similarity, old_indices[row_index],
                                   new_indices[int(col_index)]))
    candidates.sort(key=lambda entry: (-entry[0], entry[1], entry[2]))

    for similarity, old_index, new_index in candidates:
        if old_index not in matched_old and new_index not in matched_new:
            matches.append(ClauseMatch(old_clauses[old_index], new_clauses[new_index],
                                       round(similarity, 6), method))
            matched_old.add(old_index)
            matched_new.add(new_index)

    matches.sort(key=lambda match: match.old.start)
    return ClauseAlignment(
        tuple(matches),
        tuple(clause for i, clause in enumerate(old_clauses) if i not in matched_old),
        tuple(clause for i, clause in enumerate(new_clauses) if i not in matched_new),
    )


## Change detection

In [ ]:
@dataclass(frozen=True, slots=True)
class ChangeRecord:
    operation: Literal["UNCHANGED", "REPHRASED", "MODIFIED", "ADDED", "REMOVED", "UNCERTAIN"]
    old: PolicyClause | None
    new: PolicyClause | None
    signals: tuple[str, ...] = ()
    old_to_new: tuple[float, float, float] | None = None
    new_to_old: tuple[float, float, float] | None = None
    categories: tuple[str, ...] = ()


@lru_cache(maxsize=1)
def load_nli_model():
    """Load the NLI cross-encoder on first use."""
    try:
        from sentence_transformers import CrossEncoder
        return CrossEncoder("cross-encoder/nli-deberta-v3-small", max_length=512)
    except Exception as exc:
        raise PolicyDeltaError("Unable to load the NLI model.") from exc


_NUMBER_WORDS = dict(zip(
    "zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen fifteen sixteen seventeen eighteen nineteen twenty thirty forty fifty sixty seventy eighty ninety".split(),
    [*range(21), 30, 40, 50, 60, 70, 80, 90],
))
_NUMBER_PATTERN = re.compile(
    r"\b(\d+(?:\.\d+)?|" + "|".join(_NUMBER_WORDS) +
    r")\b\s*(days?|weeks?|months?|years?|hours?|minutes?|%|percent)?", re.I,
)
_NEGATION_PATTERN = re.compile(
    r"\b(?:not|never|no|without|cannot|can't|don't|doesn't|won't|isn't|aren't)\b", re.I
)
_MODAL_PATTERN = re.compile(
    r"\b(?:may|might|can|could|must|shall|will|should|only|unless|except|required|prohibited)\b", re.I
)


def _number_terms(text: str) -> tuple[tuple[str, str], ...]:
    terms = []
    for match in _NUMBER_PATTERN.finditer(text):
        value = match.group(1).lower()
        value = str(_NUMBER_WORDS.get(value, value))
        unit = (match.group(2) or "").lower().rstrip("s")
        terms.append((value, "%" if unit == "percent" else unit))
    return tuple(sorted(terms))


def change_signals(old: str, new: str) -> tuple[str, ...]:
    """Identify explicit linguistic differences requiring closer review."""
    signals = []
    for label, pattern in (("negation", _NEGATION_PATTERN),
                           ("modality", _MODAL_PATTERN)):
        if sorted(x.casefold() for x in pattern.findall(old)) != sorted(
                x.casefold() for x in pattern.findall(new)):
            signals.append(label)
    if _number_terms(old) != _number_terms(new):
        signals.append("quantity")
    return tuple(signals)


def _nli_probs(logits: np.ndarray) -> np.ndarray:
    if logits.ndim != 2 or logits.shape[1] != 3 or not np.isfinite(logits).all():
        raise PolicyDeltaError("NLI model returned invalid scores.")
    scores = np.exp(logits - logits.max(axis=1, keepdims=True))
    return scores / scores.sum(axis=1, keepdims=True)


def analyze_changes(
    alignment: ClauseAlignment, *, nli_model=None, confidence_threshold: float = 0.70,
) -> tuple[ChangeRecord, ...]:
    """Classify aligned clauses with bidirectional NLI and targeted checks."""
    if (isinstance(confidence_threshold, bool)
            or not isinstance(confidence_threshold, (float, int))
            or not math.isfinite(confidence_threshold)
            or not 0.5 < confidence_threshold < 1):
        raise PolicyDeltaError("NLI confidence threshold must be between 0.5 and 1.")

    findings: list[ChangeRecord] = []
    pending: list[tuple[ClauseMatch, tuple[str, ...]]] = []
    for match in alignment.matches:
        if match.old.normalized_text.casefold() == match.new.normalized_text.casefold():
            findings.append(ChangeRecord("UNCHANGED", match.old, match.new))
        else:
            pending.append((match, change_signals(match.old.text, match.new.text)))

    model = (nli_model if nli_model is not None else load_nli_model()) if pending else None
    pairs: list[tuple[str, str]] = []
    eligible: list[tuple[ClauseMatch, tuple[str, ...]]] = []
    for match, signals in pending:
        old, new = match.old.text, match.new.text
        too_long = len(old) > 1200 or len(new) > 1200 or len((old + new).split()) > 300
        if not too_long and getattr(model, "tokenizer", None) is not None:
            try:
                tokens = model.tokenizer(old, new, truncation=False)["input_ids"]
                too_long = len(tokens) > min(getattr(model, "max_seq_length", None) or 512, 512)
            except Exception as exc:
                raise PolicyDeltaError("Unable to validate NLI input length.") from exc
        if too_long:
            findings.append(ChangeRecord("UNCERTAIN", match.old, match.new,
                                         (*signals, "input_too_long")))
        else:
            eligible.append((match, signals))
            pairs.extend(((old, new), (new, old)))

    if pairs:
        try:
            logits = np.asarray(model.predict(pairs, batch_size=16,
                                               show_progress_bar=False), dtype=float)
        except Exception as exc:
            raise PolicyDeltaError("NLI inference failed.") from exc
        if logits.shape != (len(pairs), 3):
            raise PolicyDeltaError("NLI returned an unexpected number of predictions.")
        probabilities = _nli_probs(logits)
        for i, (match, signals) in enumerate(eligible):
            forward, reverse = probabilities[2 * i], probabilities[2 * i + 1]
            contradictory = max(forward[0], reverse[0]) >= confidence_threshold
            equivalent = min(forward[1], reverse[1]) >= confidence_threshold
            asymmetric = (forward[1] >= confidence_threshold) != (reverse[1] >= confidence_threshold)
            if signals or contradictory or asymmetric:
                operation = "MODIFIED"
            elif equivalent:
                operation = "REPHRASED"
            else:
                operation = "UNCERTAIN"
            findings.append(ChangeRecord(operation, match.old, match.new, signals,
                                         tuple(map(float, forward)), tuple(map(float, reverse))))

    findings.extend(ChangeRecord("REMOVED", clause, None)
                    for clause in alignment.unmatched_old)
    findings.extend(ChangeRecord("ADDED", None, clause)
                    for clause in alignment.unmatched_new)
    findings.sort(key=lambda finding: (
        finding.old.start if finding.old is not None else float("inf"),
        finding.new.start if finding.new is not None else float("inf"),
    ))
    return tuple(
        replace(
            finding,
            categories=classify_privacy_topics(
                *(clause.text for clause in (finding.old, finding.new)
                  if clause is not None)
            ),
        )
        for finding in findings
    )


## Privacy topics

In [ ]:
from dataclasses import replace


_TOPIC_RULES = {
    "data_collection": r"\b(?:collect\w*|gather\w*|obtain\w*|acquir\w*|capture\w*|receiv\w*)\b.{0,80}\b(?:data|information|details|identifiers?|addresses?|locations?|email|usage|device)\b|\b(?:data|information|details|identifiers?|locations?)\b.{0,80}\b(?:collect\w*|gather\w*|obtain\w*)\b",
    "data_use": r"\b(?:us(?:e|es|ed|ing)|process\w*|profil\w*|analy[sz]\w*|personaliz\w*)\b.{0,80}\b(?:data|information|activity|behavior|analytics|advertising|marketing|personalization|purposes?)\b|\b(?:data|information)\b.{0,70}\b(?:analytics|advertising|marketing|personalization)\b",
    "third_party_sharing": r"\b(?:shar\w*|disclos\w*|sell|sold|provid\w*)\b.{0,90}\b(?:data|information|details|advertisers?|partners?|third.part(?:y|ies)|affiliates?|vendors?|service providers?)\b|\b(?:third.part(?:y|ies)|advertisers?|partners?|affiliates?|vendors?|service providers?)\b.{0,90}\b(?:receiv\w*|access|disclos\w*|shar\w*)\b",
    "data_retention": r"\b(?:retain\w*|retention|stor(?:e|es|ed|ing|age)|keep|kept|delet\w*|eras\w*|remov\w*)\b.{0,80}\b(?:data|information|records?|accounts?|until|days?|months?|years?|hours?|period)\b|\b(?:data|information|records?|accounts?)\b.{0,80}\b(?:retain\w*|retention|stor(?:e|es|ed|ing|age)|delet\w*|eras\w*|kept)\b",
    "user_rights": r"\b(?:opt.?out|withdraw (?:your )?consent|data subject rights?|right to (?:access|erasure|deletion|correct|object|portability)|request (?:the )?(?:deletion|access|correction|copy)|privacy choices)\b|\b(?:you|users?|individuals?)\s+(?:may|can|have (?:the )?right to)\s+(?:request|access|delete|correct|withdraw|opt|object|view|erase|download|export)\b",
    "data_security": r"\b(?:encrypt\w*|security|secure|protect\w*|safeguard\w*|access controls?|data breaches?|authentication|unauthorized access)\b",
    "cookies_tracking": r"\b(?:cookies?|track(?:ing|ers?)|pixels?|web beacons?|advertising identifiers?|device identifiers?|session identifiers?)\b",
    "international_transfers": r"\b(?:international|cross.border|outside (?:the )?(?:eu|eea|country|region)|another country|other countries|foreign jurisdictions?)\b.{0,90}\b(?:data|information|transfer\w*|process\w*|stor\w*|servers?|countries|jurisdictions?)\b|\b(?:transfer\w*|process\w*|stor\w*)\b.{0,90}\b(?:outside (?:the )?(?:eu|eea|country|region)|overseas|other countries|foreign jurisdictions?|internationally)\b",
    "sensitive_data": r"\b(?:sensitive (?:personal )?(?:data|information)|health (?:data|information)|medical records?|biometric\w*|genetic\w*|precise (?:geolocation|location)|financial (?:data|information)|payment cards?|sexual orientation|racial|ethnic|religious beliefs?|criminal records?)\b",
    "children": r"\b(?:children|child|kids?|minors?|parental consent|under (?:the age of )?(?:13|16|18))\b",
    "policy_updates": r"\b(?:privacy policy|this policy)\b.{0,80}\b(?:updat\w*|chang\w*|modif\w*|effective|notif\w*|notice)\b|\b(?:notif\w*|updat\w*)\b.{0,80}\b(?:privacy policy|this policy)\b",
}
_TOPIC_PATTERNS = {
    category: re.compile(pattern, re.IGNORECASE)
    for category, pattern in _TOPIC_RULES.items()
}


def classify_privacy_topics(*passages: str) -> tuple[str, ...]:
    """Assign privacy topics supported by the supplied source passages."""
    if not passages or any(not isinstance(text, str) for text in passages):
        raise PolicyDeltaError("Privacy-topic classification requires text passages.")
    categories = tuple(
        category for category, pattern in _TOPIC_PATTERNS.items()
        if any(pattern.search(text) for text in passages)
    )
    return categories or ("other",)


## Validation tests

In [ ]:
class PolicyDeltaTests(unittest.TestCase):
    def setUp(self) -> None:
        self.document = PolicyDocument.from_text(
            "We retain personal data for 30 days.", role="old", document_id="sample"
        )

    def test_default_settings(self):
        self.assertEqual(Settings().seed, 42)

    def test_invalid_settings(self):
        for values in ({"seed": -1}, {"seed": True}, {"max_document_chars": 0},
                       {"max_pdf_pages": 0}, {"max_pdf_bytes": 0}):
            with self.subTest(values=values), self.assertRaises(PolicyDeltaError):
                Settings(**values)

    def test_document_checksum(self):
        expected = hashlib.sha256(self.document.text.encode("utf-8")).hexdigest()
        self.assertEqual(self.document.sha256, expected)

    def test_empty_document(self):
        with self.assertRaises(PolicyDeltaError):
            PolicyDocument.from_text("  ", role="old", document_id="empty")

    def test_input_limit(self):
        with self.assertRaises(PolicyDeltaError):
            PolicyDocument.from_text("long text", role="old", document_id="large",
                                     settings=Settings(max_document_chars=5))

    def test_invalid_role(self):
        with self.assertRaises(PolicyDeltaError):
            PolicyDocument.from_text("Valid text", role="invalid", document_id="bad")

    def test_evidence_exact_match(self):
        start = self.document.text.index("30 days")
        span = self.document.excerpt(start, start + len("30 days"))
        verify_evidence(span, self.document)
        self.assertEqual(span.text, "30 days")

    def test_tampered_evidence(self):
        span = EvidenceSpan("sample", 0, 2, "No")
        with self.assertRaises(EvidenceError):
            verify_evidence(span, self.document)

    def test_evidence_other_document(self):
        span = EvidenceSpan("another", 0, 2, "We")
        with self.assertRaises(EvidenceError):
            verify_evidence(span, self.document)

    def test_out_of_bounds(self):
        with self.assertRaises(EvidenceError):
            self.document.excerpt(0, len(self.document.text) + 1)

    def test_unicode_offsets(self):
        doc = PolicyDocument.from_text("Adatkezelés ✓", role="new", document_id="unicode")
        span = doc.excerpt(12, 13)
        self.assertEqual(span.text, "✓")
        verify_evidence(span, doc)

    def test_json_serialization(self):
        result = json.loads(document_json(self.document))
        self.assertEqual(result["role"], "old")
        self.assertEqual(result["sha256"], self.document.sha256)

    def test_generated_document_id(self):
        first = PolicyDocument.from_text("Policy text", role="old")
        second = PolicyDocument.from_text("Policy text", role="old")
        self.assertEqual(first.document_id, second.document_id)
        self.assertNotEqual(first.document_id,
                            PolicyDocument.from_text("Policy text", role="new").document_id)

    def test_txt_ingestion_preserves_offsets(self):
        content = "Data retention: 30 days.\r\nPrivacy ✓"
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "privacy.TXT"
            path.write_bytes(content.encode("utf-8"))
            document = PolicyDocument.from_txt(path, role="new")
        self.assertEqual(document.text, content)
        start = document.text.index("Privacy")
        verify_evidence(document.excerpt(start, len(document.text)), document)

    def test_txt_utf8_bom(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "privacy.txt"
            path.write_bytes("Consent required".encode("utf-8-sig"))
            document = PolicyDocument.from_txt(str(path), role="old")
        self.assertEqual(document.text, "Consent required")

    def test_txt_invalid_encoding(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "privacy.txt"
            path.write_bytes(b"\xff\xfe")
            with self.assertRaises(PolicyDeltaError):
                PolicyDocument.from_txt(path, role="old")

    def test_txt_missing_or_unsupported_file(self):
        with tempfile.TemporaryDirectory() as directory:
            for name in ("missing.txt", "policy.pdf"):
                with self.subTest(name=name), self.assertRaises(PolicyDeltaError):
                    PolicyDocument.from_txt(Path(directory) / name, role="old")

    def test_txt_character_limit(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "privacy.txt"
            path.write_text("123456", encoding="utf-8")
            with self.assertRaises(PolicyDeltaError):
                PolicyDocument.from_txt(path, role="new",
                                        settings=Settings(max_document_chars=5))

    def test_txt_rejects_invalid_path(self):
        with self.assertRaises(PolicyDeltaError):
            PolicyDocument.from_txt(None, role="old")

    def test_html_extraction_and_evidence(self):
        markup = ("<html><head><title>Website</title><script>tracker()</script></head>"
                  "<body><nav>Sign in</nav><main><h1>Privacy Policy</h1>"
                  "<p>We <strong>do not</strong> share personal data.</p>"
                  "<p>Records are kept for 30 days.</p></main>"
                  "<footer>Contact us</footer></body></html>")
        document = PolicyDocument.from_html(markup, role="new")
        self.assertEqual(document.text, "Privacy Policy\n\nWe do not share personal data."
                                          "\n\nRecords are kept for 30 days.")
        start = document.text.index("do not")
        verify_evidence(document.excerpt(start, start + 6), document)

    def test_html_without_semantic_main(self):
        markup = ("<div class='navigation'>Ignore navigation</div>"
                  "<div class='policy'><div>Collected data includes email.</div>"
                  "<div>We delete it on request.</div></div>")
        document = PolicyDocument.from_html(markup, role="old")
        self.assertEqual(document.text, "Collected data includes email.\n\n"
                                        "We delete it on request.")

    def test_html_lists_and_entities(self):
        markup = "<main><h2>Your rights</h2><ul><li>Access &amp; correction</li>" \
                 "<li>Delete your data</li></ul></main>"
        document = PolicyDocument.from_html(markup, role="old")
        self.assertEqual(document.text, "Your rights\n\nAccess & correction\n\n"
                                        "Delete your data")

    def test_html_nested_lists_and_line_breaks(self):
        markup = ("<main><ul><li>Data requests<ul><li>Deletion</li></ul></li></ul>"
                  "<p>Contact us<br>for assistance</p>"
                  "<p style='display:none'>Hidden text</p></main>")
        doc = PolicyDocument.from_html(markup, role="new")
        self.assertEqual(doc.text, "Data requests\n\nDeletion\n\nContact us\n\n"
                                   "for assistance")

    def test_html_file_ingestion(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "privacy.HTM"
            path.write_bytes("<main><p>Privacy ✓</p></main>".encode("utf-8-sig"))
            document = PolicyDocument.from_html_file(path, role="new")
        self.assertEqual(document.text, "Privacy ✓")

    def test_html_malformed_and_empty(self):
        document = PolicyDocument.from_html("<main><p>Data is retained", role="new")
        self.assertEqual(document.text, "Data is retained")
        with self.assertRaises(PolicyDeltaError):
            PolicyDocument.from_html("<script>no policy text</script>", role="old")

    def test_html_invalid_files(self):
        with tempfile.TemporaryDirectory() as directory:
            missing = Path(directory) / "missing.html"
            wrong = Path(directory) / "policy.txt"
            bad = Path(directory) / "invalid.html"
            bad.write_bytes(b"\xff\xfe")
            for path in (missing, wrong, bad):
                with self.subTest(path=path.name), self.assertRaises(PolicyDeltaError):
                    PolicyDocument.from_html_file(path, role="old")

    def test_html_size_and_type_checks(self):
        for html in (None, b"<p>Text</p>"):
            with self.subTest(html=html), self.assertRaises(PolicyDeltaError):
                PolicyDocument.from_html(html, role="old")
        with self.assertRaises(PolicyDeltaError):
            PolicyDocument.from_html("<p>too much text</p>", role="old",
                                     settings=Settings(max_document_chars=3))
        with self.assertRaises(PolicyDeltaError):
            PolicyDocument.from_html("<p>123456</p>", role="old",
                                     settings=Settings(max_document_chars=5))

    def test_pdf_ingestion_and_evidence(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "policy.PDF"
            with pymupdf.open() as pdf:
                page = pdf.new_page()
                page.insert_text((72, 72), "We retain records for 90 days.")
                pdf.save(str(path))
            document = PolicyDocument.from_pdf(path, role="new")
        self.assertIn("We retain records for 90 days.", document.text)
        start = document.text.index("90 days")
        verify_evidence(document.excerpt(start, start + len("90 days")), document)

    def test_pdf_page_order_and_empty_pages(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "policy.pdf"
            with pymupdf.open() as pdf:
                pdf.new_page().insert_text((72, 72), "Data collection")
                pdf.new_page()
                pdf.new_page().insert_text((72, 72), "Data deletion")
                pdf.save(str(path))
            document = PolicyDocument.from_pdf(path, role="old")
        self.assertEqual(document.text, "Data collection\n\nData deletion")

    def test_pdf_rejects_image_only_document(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "scan.pdf"
            with pymupdf.open() as pdf:
                pdf.new_page()
                pdf.save(str(path))
            with self.assertRaisesRegex(PolicyDeltaError, "extractable text"):
                PolicyDocument.from_pdf(path, role="old")

    def test_pdf_rejects_encrypted_document(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "protected.pdf"
            with pymupdf.open() as pdf:
                pdf.new_page().insert_text((72, 72), "Restricted")
                pdf.save(str(path), encryption=pymupdf.PDF_ENCRYPT_AES_256,
                         user_pw="secret", owner_pw="owner")
            with self.assertRaisesRegex(PolicyDeltaError, "Password-protected"):
                PolicyDocument.from_pdf(path, role="old")

    def test_pdf_invalid_paths_and_data(self):
        with tempfile.TemporaryDirectory() as directory:
            corrupt = Path(directory) / "corrupt.pdf"
            corrupt.write_bytes(b"not a PDF")
            for path in (None, Path(directory) / "missing.pdf",
                         Path(directory) / "policy.txt", corrupt):
                with self.subTest(path=path), self.assertRaises(PolicyDeltaError):
                    PolicyDocument.from_pdf(path, role="old")

    def test_pdf_resource_limits(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "policy.pdf"
            with pymupdf.open() as pdf:
                pdf.new_page().insert_text((72, 72), "Data retention is 180 days")
                pdf.new_page().insert_text((72, 72), "Data deletion")
                pdf.save(str(path))
            for settings in (
                Settings(max_pdf_pages=1),
                Settings(max_pdf_bytes=10),
                Settings(max_document_chars=5),
            ):
                with self.subTest(settings=settings), self.assertRaises(PolicyDeltaError):
                    PolicyDocument.from_pdf(path, role="new", settings=settings)

    def test_normalization_preserves_meaningful_terms(self):
        raw = "We do not share \n personal data for 180 days."
        self.assertEqual(normalize_policy_text(raw),
                         "We do not share personal data for 180 days.")

    def test_sentence_segmentation_and_exact_offsets(self):
        document = PolicyDocument.from_text(
            "We collect email addresses. We do not share them!",
            role="old")
        clauses = segment_policy(document)
        self.assertEqual(len(clauses), 2)
        self.assertEqual(clauses[1].text, "We do not share them!")
        for clause in clauses:
            self.assertEqual(document.text[clause.start:clause.end], clause.text)
            verify_evidence(document.excerpt(clause.start, clause.end), document)

    def test_headings_and_multiline_context(self):
        document = PolicyDocument.from_text(
            "Privacy Policy\n\nData Retention\nWe retain records\n"
            "for 30 days. We delete them on request.\n\n"
            "Data Sharing\nWe do not share data.", role="new")
        clauses = segment_policy(document)
        self.assertEqual([c.section for c in clauses],
                         ["Data Retention", "Data Retention", "Data Sharing"])
        self.assertEqual(clauses[0].normalized_text,
                         "We retain records for 30 days.")
        self.assertIn("\n", clauses[0].text)

    def test_abbreviations_and_quantities(self):
        document = PolicyDocument.from_text(
            "We use e.g. device IDs in the U.S. We retain them for 3.5 years. "
            "You can request deletion.", role="old")
        clauses = segment_policy(document)
        self.assertEqual(len(clauses), 3)
        self.assertIn("3.5 years", clauses[1].text)
        self.assertIn("e.g.", clauses[0].text)

    def test_bullet_items_and_paragraph_tracking(self):
        document = PolicyDocument.from_text(
            "Your Rights\n- Access your data\n- Delete your data\n\n"
            "We respond within 30 days.", role="new")
        clauses = segment_policy(document)
        self.assertEqual(len(clauses), 3)
        self.assertEqual([c.paragraph_index for c in clauses], [0, 1, 2])
        self.assertTrue(all(c.section == "Your Rights" for c in clauses))
        self.assertEqual(clauses[1].text, "- Delete your data")

    def test_html_integration(self):
        document = PolicyDocument.from_html(
            "<main><h2>Data Sharing</h2><p>We may share data. "
            "We require consent.</p></main>", role="new")
        clauses = segment_policy(document)
        self.assertEqual([c.section for c in clauses], ["Data Sharing"] * 2)
        self.assertEqual(clauses[1].text, "We require consent.")

    def test_sentence_case_headings(self):
        document = PolicyDocument.from_text(
            "Data retention\n\nWe retain information for 90 days.\n\n"
            "How we use your data\n\nWe analyze usage statistics.", role="new")
        clauses = segment_policy(document)
        self.assertEqual([clause.section for clause in clauses],
                         ["Data retention", "How we use your data"])

    def test_unpunctuated_privacy_statement_not_heading(self):
        document = PolicyDocument.from_text(
            "We collect device data\n\nWe share it with processors", role="old")
        self.assertEqual(len(segment_policy(document)), 2)

    def test_heading_only_document(self):
        document = PolicyDocument.from_text("Privacy Policy\n\nData Sharing", role="new")
        with self.assertRaisesRegex(PolicyDeltaError, "no analyzable clauses"):
            segment_policy(document)

    def test_unicode_and_crlf_offsets(self):
        document = PolicyDocument.from_text(
            "Données ✓ are stored.\r\nWe delete them after 30 days.", role="old")
        clauses = segment_policy(document)
        self.assertEqual(len(clauses), 2)
        self.assertEqual(clauses[0].start, 0)
        self.assertEqual(clauses[1].text, "We delete them after 30 days.")
        for clause in clauses:
            verify_evidence(document.excerpt(clause.start, clause.end), document)

    def test_identifiers_are_stable(self):
        document = PolicyDocument.from_text("One sentence. Another sentence.", role="new")
        self.assertEqual([c.clause_id for c in segment_policy(document)],
                         [c.clause_id for c in segment_policy(document)])

    def test_reproducible_seed(self):
        initialize(Settings(seed=7))
        first = random.random()
        initialize(Settings(seed=7))
        self.assertEqual(first, random.random())
        initialize()


    def test_alignment_reordered_clauses(self):
        old = PolicyDocument.from_text(
            "We retain data for 30 days. We share statistics with processors.", role="old")
        new = PolicyDocument.from_text(
            "We share statistics with processors. We retain data for 30 days.", role="new")
        alignment = align_policies(old, new)
        self.assertEqual([match.method for match in alignment.matches], ["exact", "exact"])
        self.assertEqual([match.new.text for match in alignment.matches],
                         ["We retain data for 30 days.",
                          "We share statistics with processors."])
        self.assertFalse(alignment.unmatched_old or alignment.unmatched_new)

    def test_alignment_rephrased_clause(self):
        old = PolicyDocument.from_text(
            "We collect email addresses for account verification.", role="old")
        new = PolicyDocument.from_text(
            "We collect email addresses for identity verification.", role="new")
        match, = align_policies(old, new).matches
        self.assertEqual(match.method, "tfidf")
        self.assertGreaterEqual(match.similarity, 0.4)
        verify_evidence(old.excerpt(match.old.start, match.old.end), old)
        verify_evidence(new.excerpt(match.new.start, match.new.end), new)

    def test_alignment_strong_paraphrase_not_forced(self):
        old = PolicyDocument.from_text(
            "We collect email addresses for account verification.", role="old")
        new = PolicyDocument.from_text(
            "We collect email addresses to verify accounts.", role="new")
        result = align_policies(old, new)
        self.assertFalse(result.matches)
        self.assertEqual(len(result.unmatched_old), 1)
        self.assertEqual(len(result.unmatched_new), 1)

    def test_alignment_additions_and_removals(self):
        old = PolicyDocument.from_text(
            "We retain records for 30 days. We never collect biometric data.", role="old")
        new = PolicyDocument.from_text(
            "We retain records for 30 days. We contact users by email.", role="new")
        result = align_policies(old, new)
        self.assertEqual(len(result.matches), 1)
        self.assertEqual(result.unmatched_old[0].text,
                         "We never collect biometric data.")
        self.assertEqual(result.unmatched_new[0].text,
                         "We contact users by email.")

    def test_alignment_duplicate_clauses_not_reused(self):
        old = PolicyDocument.from_text(
            "We delete old records. We delete old records.", role="old")
        new = PolicyDocument.from_text("We delete old records.", role="new")
        result = align_policies(old, new)
        self.assertEqual(len(result.matches), 1)
        self.assertEqual(len(result.unmatched_old), 1)
        self.assertFalse(result.unmatched_new)

    def test_alignment_threshold_can_prevent_match(self):
        old = PolicyDocument.from_text(
            "We collect email addresses to verify accounts.", role="old")
        new = PolicyDocument.from_text(
            "We collect email addresses for account verification.", role="new")
        result = align_policies(old, new, minimum_similarity=1.0)
        self.assertFalse(result.matches)
        self.assertEqual(len(result.unmatched_old), 1)
        self.assertEqual(len(result.unmatched_new), 1)

    def test_alignment_validates_arguments(self):
        old = PolicyDocument.from_text("We retain records.", role="old")
        new = PolicyDocument.from_text("We retain records.", role="new")
        for value in (0, -1, 1.1, True, float("nan"), float("inf")):
            with self.subTest(threshold=value), self.assertRaises(PolicyDeltaError):
                align_policies(old, new, minimum_similarity=value)
        for value in (0, -1, 1.5, True):
            with self.subTest(candidate_count=value), self.assertRaises(PolicyDeltaError):
                align_policies(old, new, candidates_per_clause=value)
        with self.assertRaises(PolicyDeltaError):
            align_policies(new, old)

    def test_alignment_case_whitespace_exact_match(self):
        old = PolicyDocument.from_text("We retain   records for 30 days.", role="old")
        new = PolicyDocument.from_text("we retain records for 30 days.", role="new")
        match, = align_policies(old, new).matches
        self.assertEqual(match.method, "exact")
        self.assertEqual(match.similarity, 1.0)

    def test_alignment_is_deterministic(self):
        old = PolicyDocument.from_text(
            "We share device data. We retain records for 30 days.", role="old")
        new = PolicyDocument.from_text(
            "We retain records for 90 days. We share device statistics.", role="new")
        first, second = align_policies(old, new), align_policies(old, new)
        self.assertEqual(first, second)

    def test_tfidf_similarity_uses_lexical_evidence(self):
        old = PolicyDocument.from_text("We do not share personal data.", role="old")
        new = PolicyDocument.from_text("We may share personal data.", role="new")
        scores = tfidf_similarity(segment_policy(old), segment_policy(new))
        self.assertEqual(scores.shape, (1, 1))
        self.assertGreater(scores[0, 0], 0)
        self.assertLess(scores[0, 0], 1)

    def test_tfidf_similarity_empty_vocab_and_inputs(self):
        self.assertEqual(tfidf_similarity([], []).shape, (0, 0))
        self.assertEqual(tfidf_similarity([], segment_policy(self.document)).shape,
                         (0, 1))
        old = [PolicyClause("x", "old", "...", 0, 3, None, 0)]
        new = [PolicyClause("y", "new", "!!!", 0, 3, None, 0)]
        self.assertEqual(tfidf_similarity(old, new)[0, 0], 0)

    def test_alignment_across_input_formats(self):
        old = PolicyDocument.from_html(
            "<main><h2>Data Sharing</h2><p>We disclose data to vendors.</p></main>",
            role="old")
        new = PolicyDocument.from_text(
            "Data Sharing\n\nWe disclose data to vendors.", role="new")
        result = align_policies(old, new)
        self.assertEqual(len(result.matches), 1)
        self.assertEqual(result.matches[0].method, "exact")


    def test_minilm_alignment_matches_paraphrase(self):
        class Encoder:
            def encode(self, texts, **kwargs):
                return np.array([[1.0, 0.0] if "information" in t
                                 else [2.0, 0.0] for t in texts])

        old = PolicyDocument.from_text(
            "We collect information to authenticate your account.", role="old")
        new = PolicyDocument.from_text(
            "Your details are gathered for identity confirmation.", role="new")
        self.assertFalse(align_policies(old, new).matches)
        match, = align_policies(old, new, method="minilm", encoder=Encoder()).matches
        self.assertEqual(match.method, "minilm")
        self.assertAlmostEqual(match.similarity, 1.0)
        verify_evidence(old.excerpt(match.old.start, match.old.end), old)
        verify_evidence(new.excerpt(match.new.start, match.new.end), new)

    def test_minilm_preserves_exact_priority(self):
        old = PolicyDocument.from_text("We delete records after 30 days.", role="old")
        new = PolicyDocument.from_text("We delete records after 30 days.", role="new")
        match, = align_policies(old, new, method="minilm").matches
        self.assertEqual(match.method, "exact")

    def test_minilm_empty_input_does_not_load_model(self):
        clauses = segment_policy(self.document)
        self.assertEqual(minilm_similarity([], clauses).shape, (0, 1))
        self.assertEqual(minilm_similarity(clauses, []).shape, (1, 0))

    def test_minilm_rejects_invalid_vectors(self):
        class Encoder:
            def __init__(self, output):
                self.output = output

            def encode(self, texts, **kwargs):
                return self.output

        clauses = segment_policy(self.document)
        for output in (np.array([[1.0], [float("nan")]]),
                       np.array([[0.0], [0.0]]), np.array([1.0, 2.0])):
            with self.subTest(output=output.shape), self.assertRaises(PolicyDeltaError):
                minilm_similarity(clauses, clauses, encoder=Encoder(output))

    def test_minilm_rejects_invalid_method(self):
        with self.assertRaises(PolicyDeltaError):
            align_policies(self.document, PolicyDocument.from_text(
                self.document.text, role="new"), method="unknown")

    def test_minilm_preserves_one_to_one_alignment(self):
        class Encoder:
            def encode(self, texts, **kwargs):
                return np.array([[1., 0.], [1., 0.], [1., 0.]])

        old = PolicyDocument.from_text(
            "We erase customer profiles. We retain limited transaction records.", role="old")
        new = PolicyDocument.from_text(
            "User accounts are removed from storage.", role="new")
        result = align_policies(old, new, method="minilm", encoder=Encoder())
        self.assertEqual(len(result.matches), 1)
        self.assertEqual(len(result.unmatched_old), 1)
        self.assertEqual(len(result.unmatched_new), 0)


    @staticmethod
    def _paired_changes(old_text, new_text, **kwargs):
        old = PolicyDocument.from_text(old_text, role="old")
        new = PolicyDocument.from_text(new_text, role="new")
        alignment = ClauseAlignment((ClauseMatch(segment_policy(old)[0],
                                                 segment_policy(new)[0], 0.9, "minilm"),), (), ())
        return analyze_changes(alignment, **kwargs)

    def test_changes_unchanged_without_model(self):
        old = PolicyDocument.from_text("We retain records.", role="old")
        new = PolicyDocument.from_text("We retain records.", role="new")
        result = analyze_changes(align_policies(old, new))
        self.assertEqual([entry.operation for entry in result], ["UNCHANGED"])
        self.assertIsNone(result[0].old_to_new)

    def test_changes_unmatched_clauses(self):
        old = PolicyDocument.from_text("We retain records.", role="old")
        new = PolicyDocument.from_text("We collect device identifiers.", role="new")
        result = analyze_changes(ClauseAlignment((), tuple(segment_policy(old)),
                                                  tuple(segment_policy(new))))
        self.assertEqual([entry.operation for entry in result], ["REMOVED", "ADDED"])
        verify_evidence(old.excerpt(result[0].old.start, result[0].old.end), old)
        verify_evidence(new.excerpt(result[1].new.start, result[1].new.end), new)

    def test_changes_detect_numeric_shift(self):
        self.assertEqual(change_signals("We retain data for 30 days.",
                                        "We retain data for 90 days."), ("quantity",))
        self.assertEqual(change_signals("We retain data for thirty days.",
                                        "We retain data for 30 days."), ())
        self.assertEqual(change_signals("We retain data for 30 days.",
                                        "We retain data for 30 months."), ("quantity",))

    def test_changes_detect_negation_and_modality(self):
        self.assertEqual(change_signals("We do not share information.",
                                        "We may share information."),
                         ("negation", "modality"))
        self.assertEqual(change_signals("We will store data.", "We may store data."),
                         ("modality",))

    def test_changes_equivalent_paraphrase(self):
        class Model:
            def predict(self, pairs, **kwargs):
                return np.array([[0, 7, 0], [0, 7, 0]], dtype=float)

        result, = self._paired_changes("We delete account records after thirty days.",
                                       "Account records are deleted after 30 days.",
                                       nli_model=Model())
        self.assertEqual(result.operation, "REPHRASED")
        self.assertEqual(result.signals, ())
        self.assertGreater(result.old_to_new[1], 0.9)

    def test_changes_bidirectional_asymmetry(self):
        class Model:
            def predict(self, pairs, **kwargs):
                return np.array([[0, 7, 0], [0, 0, 7]])

        result, = self._paired_changes("We collect device identifiers.",
                                       "We collect device identifiers for advertising.",
                                       nli_model=Model())
        self.assertEqual(result.operation, "MODIFIED")
        self.assertGreater(result.old_to_new[1], 0.9)
        self.assertGreater(result.new_to_old[2], 0.9)

    def test_changes_contradiction(self):
        class Model:
            def predict(self, pairs, **kwargs):
                return [[7, 0, 0], [7, 0, 0]]

        result, = self._paired_changes("We sell personal information.",
                                       "We never sell personal information.",
                                       nli_model=Model())
        self.assertEqual(result.operation, "MODIFIED")
        self.assertIn("negation", result.signals)

    def test_changes_uncertain_neutral(self):
        class Model:
            def predict(self, pairs, **kwargs):
                return [[0, 0, 7], [0, 0, 7]]

        result, = self._paired_changes("We protect information with encryption.",
                                       "We store information on secure servers.",
                                       nli_model=Model())
        self.assertEqual(result.operation, "UNCERTAIN")

    def test_changes_rule_signal_even_if_nli_entails(self):
        class Model:
            def predict(self, pairs, **kwargs):
                return [[0, 7, 0], [0, 7, 0]]

        result, = self._paired_changes("We retain records for 30 days.",
                                       "We retain records for 90 days.",
                                       nli_model=Model())
        self.assertEqual(result.operation, "MODIFIED")
        self.assertIn("quantity", result.signals)

    def test_changes_rejects_invalid_model_outputs(self):
        class Model:
            def __init__(self, value):
                self.value = value

            def predict(self, pairs, **kwargs):
                return self.value

        for output in (np.zeros((1, 3)), np.zeros((2, 2)),
                       np.full((2, 3), float("nan"))):
            with self.subTest(output=output.shape), self.assertRaises(PolicyDeltaError):
                self._paired_changes("We collect information.",
                                     "We store information.", nli_model=Model(output))

    def test_changes_validates_threshold(self):
        for value in (0.0, 0.5, 1.0, True, float("nan")):
            with self.subTest(value=value), self.assertRaises(PolicyDeltaError):
                analyze_changes(ClauseAlignment((), (), ()), confidence_threshold=value)

    def test_changes_long_clause_requires_review(self):
        class Model:
            def predict(self, pairs, **kwargs):
                raise AssertionError("Long input must not be inferred")

        old = "We collect " + "location " * 165 + "data."
        new = "We store " + "location " * 165 + "data."
        result, = self._paired_changes(old, new, nli_model=Model())
        self.assertEqual(result.operation, "UNCERTAIN")
        self.assertIn("input_too_long", result.signals)

    def test_changes_handles_tokenizer_limits(self):
        class Model:
            max_seq_length = 32

            @staticmethod
            def tokenizer(old, new, **kwargs):
                return {"input_ids": list(range(40))}

            def predict(self, pairs, **kwargs):
                raise AssertionError("Truncated input should not be inferred")

        result, = self._paired_changes("We store contact data.",
                                       "We collect contact data.", nli_model=Model())
        self.assertEqual(result.operation, "UNCERTAIN")
        self.assertIn("input_too_long", result.signals)

    def test_changes_surfaces_inference_errors(self):
        class Model:
            def predict(self, pairs, **kwargs):
                raise RuntimeError("Model unavailable")

        with self.assertRaisesRegex(PolicyDeltaError, "NLI inference failed"):
            self._paired_changes("We store contact data.", "We collect contact data.",
                                 nli_model=Model())

    def test_changes_integration_with_alignment(self):
        old = PolicyDocument.from_text("We retain records for 30 days.", role="old")
        new = PolicyDocument.from_text("We retain records for 90 days.", role="new")

        class Model:
            def predict(self, pairs, **kwargs):
                return [[0, 0, 7], [0, 0, 7]]

        findings = analyze_changes(align_policies(old, new), nli_model=Model())
        self.assertEqual(len(findings), 1)
        self.assertEqual(findings[0].operation, "MODIFIED")
        verify_evidence(old.excerpt(findings[0].old.start, findings[0].old.end), old)


    def test_privacy_topics_single_categories(self):
        examples = {
            "data_collection": "We collect account information.",
            "data_use": "We use personal information for advertising.",
            "third_party_sharing": "We disclose personal data to advertising partners.",
            "data_retention": "We retain account records for 90 days.",
            "user_rights": "You can request deletion of your information.",
            "data_security": "We encrypt sensitive records.",
            "cookies_tracking": "We use cookies for analytics.",
            "international_transfers": "We transfer data outside the EU.",
            "sensitive_data": "We collect biometric data.",
            "children": "Our service is not for children under 13.",
            "policy_updates": "We may update this privacy policy.",
        }
        for category, passage in examples.items():
            with self.subTest(category=category):
                self.assertIn(category, classify_privacy_topics(passage))

    def test_privacy_topics_multilabel(self):
        topics = classify_privacy_topics(
            "We collect device identifiers and share personal data with advertisers."
        )
        self.assertIn("data_collection", topics)
        self.assertIn("third_party_sharing", topics)
        self.assertEqual(len(topics), len(set(topics)))

    def test_privacy_topics_combine_old_and_new(self):
        categories = classify_privacy_topics(
            "We retain account records for 30 days.",
            "We share your information with third parties.",
        )
        self.assertIn("data_retention", categories)
        self.assertIn("third_party_sharing", categories)

    def test_privacy_topics_passive_and_reversed_phrasing(self):
        cases = (
            ("Personal information we collect includes location data.", "data_collection"),
            ("We receive information about device activity.", "data_collection"),
            ("We protect your data with technical safeguards.", "data_security"),
            ("Your account information is retained for two years.", "data_retention"),
        )
        for passage, category in cases:
            with self.subTest(passage=passage):
                self.assertIn(category, classify_privacy_topics(passage))

    def test_privacy_topics_no_cross_passage_matches(self):
        self.assertEqual(classify_privacy_topics("We collect.", "Information is public."),
                         ("other",))

    def test_privacy_topics_unknown_and_invalid(self):
        self.assertEqual(classify_privacy_topics("Contact us for questions."), ("other",))
        with self.assertRaises(PolicyDeltaError):
            classify_privacy_topics()
        with self.assertRaises(PolicyDeltaError):
            classify_privacy_topics(None)

    def test_privacy_topics_are_case_insensitive(self):
        self.assertIn("third_party_sharing",
                      classify_privacy_topics("WE SHARE PERSONAL DATA WITH ADVERTISERS."))

    def test_privacy_topics_integrated_on_matched_changes(self):
        old = PolicyDocument.from_text("We retain records for 30 days.", role="old")
        new = PolicyDocument.from_text("We retain records for 90 days.", role="new")

        class Model:
            def predict(self, pairs, **kwargs):
                return [[0, 0, 7], [0, 0, 7]]

        finding, = analyze_changes(align_policies(old, new), nli_model=Model())
        self.assertEqual(finding.operation, "MODIFIED")
        self.assertIn("data_retention", finding.categories)
        self.assertEqual(old.text[finding.old.start:finding.old.end], finding.old.text)
        self.assertEqual(new.text[finding.new.start:finding.new.end], finding.new.text)

    def test_privacy_topics_integrated_on_unmatched(self):
        old = PolicyDocument.from_text("We store account records for one year.", role="old")
        new = PolicyDocument.from_text("We use cookies to track visitors.", role="new")
        findings = analyze_changes(ClauseAlignment((), tuple(segment_policy(old)),
                                                  tuple(segment_policy(new))))
        self.assertEqual([finding.operation for finding in findings], ["REMOVED", "ADDED"])
        self.assertIn("data_retention", findings[0].categories)
        self.assertIn("cookies_tracking", findings[1].categories)

    def test_privacy_topics_unchanged_without_nli(self):
        old = PolicyDocument.from_text("We use cookies to track visitors.", role="old")
        new = PolicyDocument.from_text("We use cookies to track visitors.", role="new")
        finding, = analyze_changes(align_policies(old, new))
        self.assertEqual(finding.operation, "UNCHANGED")
        self.assertIn("cookies_tracking", finding.categories)

    def test_privacy_topics_priority_is_stable(self):
        passage = "We share personal data with advertisers and collect device information."
        first = classify_privacy_topics(passage)
        self.assertEqual(first, classify_privacy_topics(passage))
        self.assertEqual(first, ("data_collection", "third_party_sharing"))


suite = unittest.defaultTestLoader.loadTestsFromTestCase(PolicyDeltaTests)
result = unittest.TextTestRunner(verbosity=1).run(suite)
if not result.wasSuccessful():
    raise RuntimeError("PolicyDelta validation failed.")
print(f"Validation passed: {result.testsRun} tests")